# 02. 1D DP Problems

The state is a **single index** (or amount). Many classics: robbing houses, decoding, coins, longest increasing subsequence.

**Learning order:** HOUSE ROBBER → CIRCULAR → DECODE WAYS → COIN CHANGE → COIN CHANGE II → PERFECT SQUARES → LIS → MAX PRODUCT SUBARRAY → INTERVIEW THINKING

## 1. House Robber

Rob houses in a row, cannot rob two adjacent ones. Maximize money.

State: `dp[i]` = best using houses `0..i`. Choice at house `i`: **skip** (`dp[i-1]`) or **rob** (`dp[i-2] + nums[i]`). Keep only the last two values.

In [1]:
def rob(nums):
    prev2 = prev1 = 0                       # best up to i-2 and i-1
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print(rob([1, 2, 3, 1]), rob([2, 7, 9, 3, 1]))   # 4 12

def rob_circular(nums):                      # first and last are neighbours
    if len(nums) == 1:
        return nums[0]
    return max(rob(nums[:-1]), rob(nums[1:]))    # either skip the first or skip the last

print(rob_circular([2, 3, 2]), rob_circular([1, 2, 3, 1]), rob_circular([1, 2, 3]))   # 3 4 3

4 12
3 4 3


## 2. Decode Ways

`A=1, ..., Z=26`. Number of ways to decode a digit string?

`dp[i]` = ways to decode the first `i` characters. From `i`: use **one digit** (valid if not `'0'`) → `dp[i-1]`; use **two digits** (valid if `10..26`) → `dp[i-2]`.

In [2]:
def num_decodings(s):
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and 10 <= int(s[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]
    return dp[n]

print(num_decodings("12"), num_decodings("226"), num_decodings("06"))   # 2 3 0

2 3 0


## 3. Coin Change (minimum coins)

`dp[a]` = fewest coins to make amount `a`. For each coin `c ≤ a`: `dp[a] = min(dp[a], dp[a - c] + 1)`. Unreachable amounts stay `inf`. Greedy fails for arbitrary coins (see Greedy notebook), DP always works. Time O(amount × coins).

In [3]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != INF else -1

print(coin_change([1, 2, 5], 11), coin_change([2], 3), coin_change([1], 0))   # 3 -1 0

3 -1 0


## 4. Coin Change II (number of **combinations**)

Order must **not** matter (`1+2` and `2+1` are the same). Loop **coins on the outside**, amounts inside. Each coin is then considered once, in a fixed order, so every combination is counted once.

(If amounts were on the outside you would count **permutations**, see Combination Sum IV in the knapsack notebook.)

In [4]:
def change(amount, coins):
    dp = [1] + [0] * amount
    for c in coins:                           # coins outer => combinations
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

print(change(5, [1, 2, 5]), change(3, [2]))   # 4 0

4 0


## 5. Perfect Squares

Fewest perfect squares summing to `n`. Like coin change with coins `1, 4, 9, 16, ...`. `dp[i] = min(dp[i - s] + 1)` over squares `s ≤ i`.

In [5]:
def num_squares(n):
    dp = [0] + [float("inf")] * n
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            dp[i] = min(dp[i], dp[i - j * j] + 1)
            j += 1
    return dp[n]

print(num_squares(12), num_squares(13))   # 3 2

3 2


## 6. Longest Increasing Subsequence (LIS)

**O(n²) DP:** `dp[i]` = length of the LIS **ending at** `i` = `1 + max(dp[j])` for all `j < i` with `nums[j] < nums[i]`. Answer = `max(dp)`.

**O(n log n):** keep `tails[k]` = the **smallest possible last value** of an increasing subsequence of length `k + 1`. For each number, binary-search its place in `tails` (replace the first value `>= x`, or append). The **length** of `tails` is the answer (the list itself is not a real LIS).

In [6]:
from bisect import bisect_left

def lis_n2(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

def lis_nlogn(nums):
    tails = []
    for x in nums:
        i = bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

print(lis_n2([10, 9, 2, 5, 3, 7, 101, 18]), lis_nlogn([10, 9, 2, 5, 3, 7, 101, 18]))   # 4 4
print(lis_nlogn([0, 1, 0, 3, 2, 3]), lis_nlogn([7, 7, 7, 7]))   # 4 1

4 4
4 1


## 7. Maximum Product Subarray

A negative times a negative becomes positive, so track **both the max and the min** product ending at each position. When `x` is negative they swap roles.

In [7]:
def max_product(nums):
    best = cur_max = cur_min = nums[0]
    for x in nums[1:]:
        candidates = (x, cur_max * x, cur_min * x)
        cur_max, cur_min = max(candidates), min(candidates)
        best = max(best, cur_max)
    return best

print(max_product([2, 3, -2, 4]), max_product([-2, 0, -1]))   # 6 0

6 0


## 8. Interview thinking

**Signals:** "no two adjacent", "ways to decode/climb", "minimum coins/squares", "longest increasing", "best ending at i".

**Say:** "Let `dp[i]` be ... The last choice is either A (then `dp[i-1] + ...`) or B (then `dp[i-2] + ...`). Base cases ... O(n) time, O(1) space with two variables."

**Common mistakes:** `dp[i]` meaning unclear (ending at `i` vs up to `i`), coin change loop order (combinations vs permutations), `inf` handling, LIS: returning the last `tails` value instead of its length.

## Quick Revision

- House robber: `max(skip, rob + x)`. Circular: two runs.
- Decode: one digit (≠0) + two digits (10..26).
- Coin change: `dp[a] = min(dp[a - c] + 1)`. Combinations = coins outer.
- LIS: O(n²) ending at i, O(n log n) tails + bisect.
- Max product: track both max and min.